# 🌳 Notebook 1: Build a Merkle Tree from Scratch

**The big question:** *"Two replicas have a million keys each. How do I check whether they hold the same data, without sending all of it over the network?"*

A **Merkle tree** is a binary tree where:
- Every **leaf** is the hash of one piece of data.
- Every **internal node** is the hash of its two children.
- The **root** is one tiny number (32 bytes for SHA-256) that depends on *every* leaf.

Change a single byte anywhere in the data and the root hash changes completely. That's the magic: comparing one 32-byte number tells you whether two large datasets are identical.

## Learning objectives
- Implement a Merkle tree with `hashlib`.
- See that any change → different root.
- Learn why **leaf order** and **canonical serialization** matter.
- Visualize a small tree.
- Peek at "sharp edges" (domain separation / second-preimage) used by production trees.

We'll use the data structure from this notebook in Notebook 2 (inclusion proofs) and Notebook 3 (replica diff).

## 🛠️ Setup

```bash
cd 02-distributed-primitives/merkle-trees
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, run `Cmd+Shift+P` → *Reload Window*.

## 1. Hash helpers with **domain separation**

Production Merkle trees add a tiny prefix byte so that **leaf hashes** and **internal-node hashes** use different "domains":

- `H_leaf(x) = sha256(0x00 || x)`
- `H_node(l, r) = sha256(0x01 || l || r)`

Why? Without this, an attacker could sometimes feed the hash of an internal node *as if it were a leaf* and trick a verifier — a classic *second-preimage attack* on naive Merkle trees (see Certificate Transparency RFC 6962 §2.1). Adding the prefix byte closes that door.

For a beginner it's enough to remember: **use different hash recipes for leaves vs nodes**. Everything else in this lab uses these helpers.

In [ ]:
import hashlib

def H_leaf(data: bytes) -> bytes:
    return hashlib.sha256(b"\x00" + data).digest()

def H_node(left: bytes, right: bytes) -> bytes:
    return hashlib.sha256(b"\x01" + left + right).digest()

# Sanity check: identical input → identical output, deterministic.
print(H_leaf(b'hello').hex()[:16], '...')
print(H_node(H_leaf(b'a'), H_leaf(b'b')).hex()[:16], '...')

assert H_leaf(b'hello') == H_leaf(b'hello')          # deterministic
assert H_leaf(b'hello') != H_leaf(b'hell0')          # sensitive to input

# Domain separation is the point of the prefix byte, so check it does its job:
# hashing some bytes AS A LEAF must never equal hashing them AS A NODE.
x, y = H_leaf(b'a'), H_leaf(b'b')
assert H_leaf(x + y) != H_node(x, y), 'leaf and node domains collide — prefix is not working'
# Concretely: an internal node's hash can never be passed off as a leaf's hash.
internal = H_node(x, y)
assert all(H_leaf(candidate) != internal for candidate in (x + y, x, y, b''))
print('✔ leaf and internal hashes live in separate domains')

## 2. Build the tree

We hash each leaf, then repeatedly pair up adjacent hashes and hash each pair, until only one hash remains — the **root**.

When a level has an **odd** number of nodes, we duplicate the last one so the next level has a clean pair. (This is how Bitcoin does it; other systems like Certificate Transparency promote the odd node instead. Either works; just pick one and stick with it.)

In [ ]:
def build_merkle(leaves: list[bytes]) -> list[list[bytes]]:
    """Return a list of levels: levels[0] = leaf hashes, levels[-1] = [root]."""
    if not leaves:
        # Convention: empty tree has the hash of an empty leaf as its root.
        return [[H_leaf(b'')]]
    level = [H_leaf(x) for x in leaves]
    levels = [level]
    while len(level) > 1:
        if len(level) % 2 == 1:
            level = level + [level[-1]]  # duplicate last to pad
        nxt = [H_node(level[i], level[i + 1]) for i in range(0, len(level), 2)]
        levels.append(nxt)
        level = nxt
    return levels

def root_of(leaves):
    return build_merkle(leaves)[-1][0]

data = [b'alice=100', b'bob=50', b'carol=75', b'dave=20']
levels = build_merkle(data)
for i, lvl in enumerate(levels):
    print(f'level {i} ({len(lvl)} nodes): {[h.hex()[:10] for h in lvl]}')
print('\nroot:', root_of(data).hex())

## 3. One bit-flip → brand-new root

This is the foundation of every tamper-evident system: Git commits, blockchains, signed transparency logs. Flip a byte anywhere in the dataset and the root is unrecognizable.

In [ ]:
data2 = [b'alice=100', b'bob=51', b'carol=75', b'dave=20']  # bob lost a dollar
print('root before:', root_of(data).hex())
print('root after :', root_of(data2).hex())
print('equal?      :', root_of(data) == root_of(data2))

assert root_of(data) != root_of(data2)
# "Completely different", not "slightly different": about half the bits should flip
# for a one-character change. That avalanche is what makes the root a useful summary.
a, b = root_of(data), root_of(data2)
differing_bits = sum(bin(x ^ y).count('1') for x, y in zip(a, b))
print(f'bits changed in the 256-bit root: {differing_bits}')
assert 90 < differing_bits < 166, differing_bits

## 4. Order matters — pick a canonical ordering

A Merkle tree is a **sequence** of leaves. Same items in a different order → different root. If two replicas iterate keys in different orders, they'll compute different roots even with identical data.

**Fix:** before hashing, sort the leaves by some canonical key (usually the key name, or the key's hash).

In [ ]:
data_order_A = [b'alice=100', b'bob=50']
data_order_B = [b'bob=50', b'alice=100']
print('A root:', root_of(data_order_A).hex()[:16], '...')
print('B root:', root_of(data_order_B).hex()[:16], '...')
print('same?  :', root_of(data_order_A) == root_of(data_order_B))

# Canonical fix: sort before building.
def canonical_root(items: dict[str, str]) -> bytes:
    leaves = [f'{k}={v}'.encode() for k, v in sorted(items.items())]
    return root_of(leaves)

A = canonical_root({'alice': '100', 'bob': '50'})
B = canonical_root({'bob': '50', 'alice': '100'})
print('\nAfter canonical sort:')
print('A root:', A.hex()[:16], '...')
print('B root:', B.hex()[:16], '...')
print('same?  :', A == B)

assert root_of(data_order_A) != root_of(data_order_B), 'order must matter before sorting'
assert A == B, 'canonical sorting must make the roots agree'
print('\n✔ identical data in different iteration orders now produces one root')

## 5. Canonical serialization

We used `f'{k}={v}'.encode()`. That's fine for a toy, but real systems pick an **unambiguous** encoding so two independent implementations can't disagree. Examples:

- Bitcoin hashes the raw transaction bytes.
- Certificate Transparency uses TLS-style length-prefixed encoding.
- Git hashes a specific header + payload format (`blob <size>\0<bytes>`).

The rule of thumb: **there must be exactly one way to encode each input**. Otherwise two honest nodes can compute different roots.

## 6. Visualize a small tree

Here's an ASCII picture of the 4-leaf tree built from `data`. Each box shows the first 8 hex chars of that node's hash.

In [ ]:
def ascii_tree(levels):
    # Pretty-print the tree bottom-up as rows.
    max_width = len(levels[0]) * 12
    for lvl in reversed(levels):
        row = '  '.join(h.hex()[:8] for h in lvl)
        print(row.center(max_width))
        print(' ' * (max_width // 2))

ascii_tree(build_merkle(data))

## 🧪 Why a tree, not just `hash(everything)`?

A single hash of all the data also detects differences. But a tree gives you something extra:

1. When two roots differ, you can **walk down** to find *which* leaves differ in `O(log N)` steps — we'll do that in **Notebook 3**.
2. You can hand someone an `O(log N)` **proof** that a single leaf is in the tree, without showing them the rest — we'll do that in **Notebook 2**.

## ✅ Recap
- A Merkle tree summarizes *N* leaves into a 32-byte root.
- Use **domain separation** (`0x00` for leaves, `0x01` for nodes) to avoid confusion attacks.
- Define a **canonical order** and **canonical serialization** — otherwise two honest replicas get different roots.
- One bit change → completely different root.